# Polynomial CSS codes with cyclic physical types

`CyclicTypeCode` takes coefficients of scalar block polynomials over a binary abelian group
algebra. The logical group `G` and the additional physical cycle are separate choices.
`block_coefficients[a][b][d]` is the coefficient of the physical-cycle variable to power `d`.
A polynomial check row represents one full orbit under `G`.

The constructor validates the full cyclic check space and an independent orbit basis, then
validates a supplied common logical frame or attempts to find one. It does not certify distance.


In [1]:
import numpy as np

from qldpc import abstract, codes
from qldpc.objects import Pauli

ring = abstract.GroupRing(abstract.CyclicGroup(4))
x = ring.generators[0]
zero, one = ring.zero, ring.one
q = 7
# a(x,y) = 1 + y**3 + x*y**2 + x*y**4
# Coefficient index is the exponent of y, the physical-cycle variable.
a = [one, zero, x, one, x, zero, zero]
a_dagger = [a[(-d) % q].T for d in range(q)]
star = codes.CyclicTypeCode([[a, a_dagger]], check_rows=range(q - 1))
assert (star.num_qubits, star.dimension, star.get_weight()) == (56, 8, 8)
print(star)
print(star.logical_frame_result.message)

CyclicTypeCode on 56 qubits with 2 physical cycles of length 7, 2 logical orbits, and logical group CyclicGroup
Found a compatible J with 2 logical orbits (fiber shortcut).


## Coordinates and explicit logical representatives

Physical columns have order `(cycle, position, group_index)` with group index varying fastest.
Group indices follow `tuple(code.group.generate())`. Logical rows have order
`(logical_orbit, group_index)`. Translation in `G` permutes each logical orbit regularly.

The independent checks span the entire cyclic parent. Dropping rows here removes redundancy;
it does not omit stabilizers as an active-row GALA construction can do. The additional physical
cycle preserves the stabilizer space, even if it does not permute the selected rows as a list.
Its logical action is not assumed to be trivial in the general family.

An explicit polynomial `J` selects particular representatives and bypasses automatic solving.
Its coefficients must use the same ring as the checks.


In [2]:
j = abstract.RingArray.build([[1] * q + [0] * q, [0] * q + [1] * q], ring)
explicit = codes.CyclicTypeCode([[a, a_dagger]], logical_ops=j, check_rows=range(q - 1))
assert np.array_equal(explicit.get_logical_ops(Pauli.X), j.regular_lift())
assert np.array_equal(explicit.get_logical_ops(Pauli.Z), j.regular_lift())
print(explicit.logical_frame_result.message)

Validated the supplied compatible J.


## One physical cycle and a noncyclic logical group

This seven-position polynomial produces one regular logical orbit. For constant coefficients,
the code consists of independent Steane copies, one at each group site. The use of the regular
representation is deliberate, including when the supplied group has a different default lift.


In [3]:
group = abstract.AbelianGroup(2, 2, direct_sum=True)
ring = abstract.GroupRing(group)
coefficients = abstract.RingArray.build([[[1, 1, 1, 0, 1, 0, 0]]], ring)
steane_copies = codes.CyclicTypeCode(coefficients, check_rows=[0, 1, 2])
assert (len(steane_copies), steane_copies.dimension) == (28, 4)
print(steane_copies)

CyclicTypeCode on 28 qubits with 1 physical cycles of length 7, 1 logical orbits, and logical group AbelianGroup(2,2)


## A general coefficient solve and its outcomes

The fiber shortcut is not applicable to the following `q=1` example. The general solver reduces
check/logical commutation and augmentation to an affine binary system, then checks orthonormality
on its coefficient assignments. This reference method is exponential. Its default limit of
10,000 candidates bounds enumeration, not wall-clock time.

Success yields `found`. Complete exhaustion or a proved obstruction yields `infeasible`.
Stopping before exhausting the domain yields `inconclusive`; that is not a nonexistence proof.
Failures raise `LogicalFrameError`, with the structured outcome available as `.result`.


In [4]:
ring = abstract.GroupRing(abstract.CyclicGroup(3))
x = ring.generators[0]
blocks = [[[ring.one], [x], [ring.zero]]]
solved = codes.CyclicTypeCode(blocks)
assert (len(solved), solved.dimension) == (9, 3)
assert solved.logical_frame_result.method == "coefficients"
print(solved.logical_frame_result.message)

try:
    codes.CyclicTypeCode(blocks, logical_search_limit=0)
except codes.LogicalFrameError as error:
    assert error.result.status == "inconclusive"
    print(error.result.status, error.result.message)

# An explicit frame succeeds even when enumeration is disabled.
j = abstract.RingArray.build([[ring.one, x, ring.one]], ring)
assert codes.CyclicTypeCode(blocks, j, logical_search_limit=0).dimension == 3

Found a compatible J after testing 1 coefficient candidates.
inconclusive No compatible J found within 0 candidates; existence remains undecided. Increase logical_search_limit or supply logical_ops explicitly.


In [5]:
binary_ring = abstract.GroupRing(abstract.TrivialGroup())
impossible = abstract.RingArray.build([[[1, 1, 1, 1]]], binary_ring)
try:
    codes.CyclicTypeCode(impossible, check_rows=[0], logical_search_limit=None)
except codes.LogicalFrameError as error:
    assert error.result.status == "infeasible"
    print(error.result.status, error.result.message)

infeasible No compatible J exists; all candidates in the affine solution space failed.


These last checks still define an ordinary CSS code, but all commuting supports have even
weight, so an orthonormal common logical frame cannot exist. The constructor does not silently
substitute a generic binary logical basis.

## Overlapping frames and an explicit check basis

Polynomial row combinations can replace selected parent rows when they span exactly the same
check module. Logical representatives may also be changed by stabilizers while preserving all
required identities, including augmentation. The example below creates overlap between translates.


In [6]:
ring = abstract.GroupRing(abstract.CyclicGroup(4))
x = ring.generators[0]
coefficients = abstract.RingArray.build([[[1, 1, 1, 0, 1, 0, 0]]], ring)
base = codes.CyclicTypeCode(coefficients, check_rows=[0, 1, 2])
h = abstract.RingArray.build(base.check_basis, ring)
h[0] = h[0] + x * h[1]
j = abstract.RingArray.build(base.logical_prototypes, ring)
j[0] = j[0] + (ring.one + x + x**2 + x**3) * base.check_basis[0]
changed = codes.CyclicTypeCode(coefficients, j, check_basis=h)
assert np.array_equal(changed.matrix_x.row_space(), base.matrix_x.row_space())
assert np.any(np.logical_and(*changed.get_logical_ops(Pauli.X)[:2]))
print(changed.logical_frame_result.message)

Validated the supplied compatible J.
